In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from panel_utils import ModelResultsAggregator, run_panel_regressions, run_spec_tests, run_panel_model_diagnostics
from test_fun import *
import pickle
from pathlib import Path
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('cons_reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')

# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])
 
# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)

df_reg = df_reg.sort_values(['Region', 'Date']).copy()

df_reg['Cluster_3'] = ((df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)).astype(int)


# Взаимодействия общего шока
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock'] * df_reg['Sank_dum']


# Взаимодействия негативного шока 
if 'd_Mon_Shock_neg' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_1']
if 'd_Mon_Shock_neg' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_2']
if 'd_Mon_Shock_neg' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_3']
if 'd_Mon_Shock_neg' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_neg'] * df_reg['Covid_dum']
if 'd_Mon_Shock_neg' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_neg'] * df_reg['Sank_dum']

# Взаимодействия позитивного шока 
if 'd_Mon_Shock_pos' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_1']
if 'd_Mon_Shock_pos' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_2']
if 'd_Mon_Shock_pos' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_3']
if 'd_Mon_Shock_pos' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_pos'] * df_reg['Covid_dum']
if 'd_Mon_Shock_pos' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_neg'] * df_reg['Sank_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[df_reg['Cluster_3'] == 1].copy()





################
# Убираем лишнее
################

cols_to_drop = ['Num_reg', 'Ent_conf_ind_manufactoring', 'Key_Rate', 'd_Key_Rate',
        'Ent_conf_ind_manufactoring_adj', 'Nominal_Percent_Rate', 'Ent_conf_ind_mining', 'Ent_conf_ind_mining_adj',
        'MaP_Tight_Announcement', 'MaP_Ease_Announcement', 'MaP_Tight_Fact', 'MaP_Ease_Fact', 'Cluster_3', 'D_top5_rozn',]

df_reg.drop(columns = cols_to_drop, axis = 1, inplace = True)

# Модель

In [3]:
###########################
# Загрузить модель из файла
###########################

# models = load_models()
# model_name = "Модель 1"
# saved_spec = models.get(model_name)
# if saved_spec:
#     dependent_var = saved_spec["dependent_var"]
#     exog_vars_base = saved_spec["exog_vars_base"]
#     shock_vars = saved_spec["shock_vars"]
#     model_spec = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
#     exog_vars_small_1, exog_vars_small_2, exog_vars_small_3 = model_spec["exog_variants"]


### Создание модели

In [4]:
# Разделим MIACR на позитивный и негативный
df_reg['d_MIACR'] = df_reg.groupby('Region')['MIACR'].diff()
df_reg['d_MIACR_neg'] = df_reg['d_MIACR'].where(df_reg['d_MIACR'] < 0, 0)
df_reg['d_MIACR_pos'] = df_reg['d_MIACR'].where(df_reg['d_MIACR'] > 0, 0)

# # Разность монетарного шока
# df_reg['d_Mon_Shock_neg'] = df_reg.groupby('Region')['Mon_Shock_neg'].diff()
# df_reg['d_Mon_Shock_pos'] = df_reg.groupby('Region')['Mon_Shock_pos'].diff()

# Разделим ROISFIX на позитивный и негативный
df_reg['d_ROISFIX'] = df_reg.groupby('Region')['ROISFIX'].diff()
df_reg['d_ROISFIX_neg'] = df_reg['d_ROISFIX'].where(df_reg['d_ROISFIX'] < 0, 0)
df_reg['d_ROISFIX_pos'] = df_reg['d_ROISFIX'].where(df_reg['d_ROISFIX'] > 0, 0)

# # Разность монетарного шока
# df_reg['d_Mon_Shock_neg'] = df_reg.groupby('Region')['Mon_Shock_neg'].diff()
# df_reg['d_Mon_Shock_pos'] = df_reg.groupby('Region')['Mon_Shock_pos'].diff()

In [5]:
df_reg.columns

Index(['Region', 'Date', 'd_Int_Rate_ConsCred', 'Int_Rate_ConsCred',
       'Int_Rate_ConsCred_lag1', 'New_Loans_ConsCred',
       'New_Loans_ConsCred_mom', 'New_Loans_Fl', 'New_Loans_Fl_mom',
       'Zadolg_ConsCred', 'Zadolg_ConsCred_mom', 'Zadolg_Fl', 'Zadolg_Fl_mom',
       'Def_Zadolg_ConsCred', 'Def_Zadolg_Fl', 'Cred_nagr', 'Fin_Dostup',
       'Zakred', 'Exc_rate', 'Bonds_Rate_Correct_5Y', 'Inflation_Expectations',
       'Mon_Shock', 'd_Mon_Shock', 'd_Mon_Shock_neg', 'd_Mon_Shock_pos',
       'ROISFIX', 'MIACR', 'Covid_dum', 'Sank_dum', 'Cluster_1', 'Cluster_2',
       'Inflation_Expectations_adj', 'd_Inflation_Expectations',
       'CAR_Indicator', 'D_Progr_mort', 'd_Ex_Rate', 'Cap_to_assets', 'CPI',
       'IBC_constr_fed', 'IBC_torg_fed', 'IBC_auto_fed', 'd_CPI',
       'd_IBC_constr_fed', 'd_IBC_torg_fed', 'd_IBC_auto_fed',
       'd_IBC_constr_fed_adj', 'd_IBC_torg_fed_adj', 'd_IBC_auto_fed_adj',
       'd_Mon_Shock_Cl1', 'd_Mon_Shock_Cl2', 'd_Mon_Shock_Cl3',
       'd_Mon

In [6]:
# Сортировка
df_reg = df_reg.sort_values(['Region', 'Date']).copy()

# Переменные и их лаги
lag1_map = {
    "d_Int_Rate_ConsCred": "d_Int_Rate_ConsCred_lag1",
    "New_Loans_ConsCred": "New_Loans_ConsCred_lag1",
    "New_Loans_ConsCred_mom": "New_Loans_ConsCred_mom_lag1",
    "New_Loans_Fl": "New_Loans_Fl_lag1",
    "New_Loans_Fl_mom": "New_Loans_Fl_mom_lag1",
    "Zadolg_ConsCred": "Zadolg_ConsCred_lag1",
    "Zadolg_ConsCred_mom": "Zadolg_ConsCred_mom_lag1",
    "Zadolg_Fl": "Zadolg_Fl_lag1",
    "Zadolg_Fl_mom": "Zadolg_Fl_mom_lag1",
    "Def_Zadolg_ConsCred": "Def_Zadolg_ConsCred_lag1",
    "Def_Zadolg_Fl": "Def_Zadolg_Fl_lag1",
    "Zakred": "Zakred_lag1",
    "Cred_nagr": "Cred_nagr_lag1",
    "Cap_to_assets": "Cap_to_assets_lag1",
    "CAR_Indicator": "CAR_Indicator_lag1",
    "d_Ex_Rate": "d_Ex_Rate_lag1",
    "CPI": "CPI_lag1",
    "d_CPI": "d_CPI_lag1",
    "Inflation_Expectations_adj": "Inflation_Expectations_adj_lag1",
}

# Лаги переменных 
for base_col, lag_col in lag1_map.items():
    if base_col in df_reg.columns:
        df_reg[lag_col] = df_reg.groupby('Region')[base_col].shift(1)

# Лаги шоков
for shock_col in ["d_Mon_Shock_pos", "d_Mon_Shock_neg"]:
    if shock_col in df_reg.columns:
        for k in range(1, 6):
            df_reg[f"{shock_col}_lag{k}"] = df_reg.groupby('Region')[shock_col].shift(k)



In [7]:
###############################
# Создание аргументов модели
###############################

dependent_var = 'd_Int_Rate_ConsCred'

exog_vars_base = [
    'd_Int_Rate_ConsCred_lag1',             # Лаг зависимой
    'New_Loans_ConsCred_lag1',              # Показатели портфеля
    # 'Zadolg_ConsCred_lag1',
    'Def_Zadolg_ConsCred_lag1', 
    # 'Cred_nagr_lag1',                    # Закредитованность / Кредитная нагрузка
    'Zakred_lag1',                        
    'Fin_Dostup',                           # Доля фин орг / Доля топ-5
    # 'D_top5_rozn',                  
    # 'CAR_Indicator_lag1',                   # CAR / Капитал к активам / Ставка по облигациям
    'Cap_to_assets_lag1',
    # 'Bonds_Rate_Correct_5Y',                # Ставка по облигациям
    # 'Exc_rate',                             # Валютный курс
    'd_Ex_Rate',   
    # 'CPI',                                  # Инфляция
    'CPI_lag1',
    # 'd_CPI_lag1',      
    
    # 'Inflation_Expectations',               # Инфл ожидания
    # 'Inflation_Expectations_adj',
    # 'Inflation_Expectations_adj_lag1',
    # 'd_Inflation_Expe1ctations',     
    'Covid_dum',                            # Дамми
    'Sank_dum',
    'Cluster_1',
    'Cluster_2',
]

shock_vars = [
    # ['d_Mon_Shock_pos', 'd_Mon_Shock_neg'],
    ['d_Mon_Shock_pos_lag1', 'd_Mon_Shock_neg_lag1'],
    # ['d_Mon_Shock_pos_lag2', 'd_Mon_Shock_neg_lag2'],
    # ['d_Mon_Shock_pos_lag3', 'd_Mon_Shock_neg_lag3'],
    # ['d_Mon_Shock_pos_lag4', 'd_Mon_Shock_neg_lag4'],
    # ['d_Mon_Shock_pos_lag5', 'd_Mon_Shock_neg_lag5'],
    # ['d_Mon_Shock_pos_lag6', 'd_Mon_Shock_neg_lag6'],
    ['d_ROISFIX_pos', 'd_ROISFIX_neg'],
    ['d_MIACR_pos', 'd_MIACR_neg'],
]

model_spec = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
exog_vars_small_1, exog_vars_small_2, exog_vars_small_3 = model_spec["exog_variants"]


### Мультиколлинеарность

In [8]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 1
###############################

# Tips to colinearity:
# 'Bonds_Rate_Correct_5Y' with 'CAR_Indicator'
# 'Bonds_Rate_Correct_5Y' and 'CAR_Indicator' with 'ROISFIX' and 'MIACR'
# 'Cred_nagr' with 'Zakred'
# 'Zadolg_ConsCred' with 'New_Loans_ConsCred'

vif_variables = [item for item in exog_vars_small_1 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred']]
df_vif = df_reg[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1")
print("="*60)
print(vif_data.to_string(index=False))

exog_vars_small_1 = [item for item in exog_vars_small_1 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred']]
exog_vars_small_2 = [item for item in exog_vars_small_2 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred', 'Bonds_Rate_Correct_5Y']]
exog_vars_small_3 = [item for item in exog_vars_small_3 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred', 'Bonds_Rate_Correct_5Y']]


РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1
                Variable      VIF
                Sank_dum 2.595905
               d_Ex_Rate 2.361147
                CPI_lag1 2.215689
               Cluster_2 1.803632
               Covid_dum 1.755641
               Cluster_1 1.682686
    d_Mon_Shock_pos_lag1 1.655061
              Fin_Dostup 1.645192
    d_Mon_Shock_neg_lag1 1.562587
 New_Loans_ConsCred_lag1 1.484473
      Cap_to_assets_lag1 1.298093
Def_Zadolg_ConsCred_lag1 1.241523
             Zakred_lag1 1.169178
d_Int_Rate_ConsCred_lag1 1.037108


In [9]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

dependent_var = model_spec["dependent_var"]
exog_vars_initial = exog_vars_small_1

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg', intercept=False 
)

# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2379
Estimator:                   PooledOLS   R-squared (Between):              0.5340
No. Observations:                 5852   R-squared (Within):               0.2362
Date:                 Mon, Jan 26 2026   R-squared (Overall):              0.2379
Time:                         01:45:22   Log-likelihood                -1.222e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      130.16
Entities:                           77   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                 F(14,5838)
Min Obs:                        76.000                          

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1139: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Covid_dum, Sank_dum, Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                         RandomEffects Estimation Summary                        
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2379
Estimator:               RandomEffects   R-squared (Between):              0.5340
No. Observations:                 5852   R-squared (Within):               0.2362
Date:                 Mon, Jan 26 2026   R-squared (Overall):              0.2379
Time:                         01:45:22   Log-likelihood                -1.222e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      130.16
Entities:                           77   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                 F(14,5838)
Min Obs:                        76.000                                           
Max Obs:                        76.000   F-statistic (robust):             17.443
                

In [10]:
# run_panel_model_diagnostics(
#     y, X, pooled_res, fe_res, re_res,
#     pooled_success, fe_success, re_success
# )


In [11]:
# # ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

# run_spec_tests(
#     y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
# )


In [12]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

dependent_var = model_spec["dependent_var"]
exog_vars_initial = exog_vars_small_2

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА) - ROISFIX
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2145
Estimator:                   PooledOLS   R-squared (Between):             -8.1912
No. Observations:                 5852   R-squared (Within):               0.2170
Date:                 Mon, Jan 26 2026   R-squared (Overall):              0.2145
Time:                         01:45:23   Log-likelihood                -1.229e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      113.82
Entities:                           77   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                 F(14,5837)
Min Obs:                        76.000                

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1139: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Covid_dum, Sank_dum, Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                         RandomEffects Estimation Summary                        
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2145
Estimator:               RandomEffects   R-squared (Between):             -8.1912
No. Observations:                 5852   R-squared (Within):               0.2170
Date:                 Mon, Jan 26 2026   R-squared (Overall):              0.2145
Time:                         01:45:23   Log-likelihood                -1.229e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      113.82
Entities:                           77   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                 F(14,5837)
Min Obs:                        76.000                                           
Max Obs:                        76.000   F-statistic (robust):             40.128
                

In [13]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

dependent_var = model_spec["dependent_var"]
exog_vars_initial = exog_vars_small_3

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg', 
)

# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2114
Estimator:                   PooledOLS   R-squared (Between):             -7.8943
No. Observations:                 5852   R-squared (Within):               0.2138
Date:                 Mon, Jan 26 2026   R-squared (Overall):              0.2114
Time:                         01:45:23   Log-likelihood                -1.231e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      111.77
Entities:                           77   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                 F(14,5837)
Min Obs:                        76.000                          

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1139: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Covid_dum, Sank_dum, Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                         RandomEffects Estimation Summary                        
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.2114
Estimator:               RandomEffects   R-squared (Between):             -7.8943
No. Observations:                 5852   R-squared (Within):               0.2138
Date:                 Mon, Jan 26 2026   R-squared (Overall):              0.2114
Time:                         01:45:23   Log-likelihood                -1.231e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      111.77
Entities:                           77   P-value                           0.0000
Avg Obs:                        76.000   Distribution:                 F(14,5837)
Min Obs:                        76.000                                           
Max Obs:                        76.000   F-statistic (robust):             41.183
                

In [ ]:
# # Optional: save model specs to Models.pkl
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 1")
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 2")
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 3")


### Вывод результатов

In [ ]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_ConsCred'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель(м) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель(м) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель(м) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель(б) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель(б) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель(б) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_test.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

In [ ]:
models_dict = load_models()
models_table = pd.DataFrame.from_dict(models_dict, orient='index')
models_table


### Выгрузка всех моделей


In [ ]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

models_dict = load_models()

model_specs_all = []
for model_name, saved_spec in models_dict.items():
    model_spec = build_shock_variants(
        saved_spec["dependent_var"],
        saved_spec["exog_vars_base"],
        saved_spec["shock_vars"],
    )
    exog_variants = model_spec["exog_variants"]

    for idx, exog_vars_initial in enumerate(exog_variants):
        shock_entry = saved_spec["shock_vars"][idx]
        if isinstance(shock_entry, (list, tuple)):
            shock_name = "+".join(shock_entry)
        else:
            shock_name = str(shock_entry)

        dependent_var = model_spec["dependent_var"]
        y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
            df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
        )

        model_specs_all.append({
            'spec_name': f"{model_name} - {shock_name}",
            'dependent_var': dependent_var,
            'subsample': 'Общая выборка',
            'results': {
                'pooled': pooled_res if pooled_success else None,
                'fe': fe_res if fe_success else None,
                're': re_res if re_success else None
            }
        })

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

results_dir = ensure_results_dir('Results')
date_tag = pd.Timestamp.today().strftime('%d_%m_%y')
out_all = os.path.join(results_dir, f"all_models_{date_tag}.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)
